## 1) Convert MIMIC into MEDS

In [ ]:
!conda create -y -n venv_meds_mimic python=3.11

In [ ]:
cohort = "MEDS_cohort"
do_download = "True"
# physionet credentials
username = ""
password = ""

In [ ]:
!conda run -n venv_meds_mimic bash run_meds_etl.sh \
"{cohort}" "{username}" "{password}" "{do_download}"

## 2) Generate prediction labels for each task

In [ ]:
import os

TASKS_PATH = "tasks"
MEDS_cohort = "MEDS_cohort"

for f in os.listdir(TASKS_PATH):
    f_name = os.path.splitext(f)[0]
    !aces-cli \
        config_path="{TASKS_PATH}/{f}" \
        cohort_name="{f_name}" \
        cohort_dir="{MEDS_cohort}/labels" \
        data=sharded \
        data.standard=meds \
        data.root="{MEDS_cohort}/data" \
        data.shard=$(expand_shards train/292 tuning/37 held_out/37) \
        -m

## 3) Preprocessing pipeline & MEDS-KG conversion

In [ ]:
import yaml
import os
import polars as pl


def init_dirs(task_name: str, index: int):
    export_dir = f"exports/{task_name}"
    outcomes_dir = f"{export_dir}/labels"
    meds_cohort_dir = f"{export_dir}/meds/{index}/MEDS_cohort"
    os.makedirs(export_dir, exist_ok=True)
    os.makedirs(outcomes_dir, exist_ok=True)
    os.makedirs(meds_cohort_dir, exist_ok=True)
    os.makedirs(f"{export_dir}/meds/{index}/MEDS_cohort/metadata", exist_ok=True)

    return export_dir, outcomes_dir, meds_cohort_dir


with open("experiments.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
from utils.preprocessing import (
    remove_long_text,
    birthdate_to_age,
    window_dict,
    parse_blood_pressure_values,
    regenerate_ids,
    meds_core_columns,
)


def build_events_base(events, sample, scfg):
    base = (
        events.join(sample.lazy(), on="subject_id", how="inner")
        .with_columns(remove_long_text)
        .with_columns(birthdate_to_age)
        .filter(window_dict[scfg["window"]])
    )

    base = (
        parse_blood_pressure_values(base)
        .with_columns(regenerate_ids)
        .select(meds_core_columns)
    )

    return base


def split_event_types(events_base, scfg):
    static_codes = (
        events_base.filter(pl.col("time").is_null() & pl.col("numeric_value").is_null())
        .sort(["subject_id", "code"])
        .unique(subset=["subject_id", "code"], keep="last")
    )

    static_numeric = (
        events_base.filter(
            pl.col("time").is_null() & pl.col("numeric_value").is_not_null()
        )
        .sort(["subject_id", "code"])
        .unique(subset=["subject_id", "code"], keep="last")
    )

    dynamic_numeric = (
        events_base.filter(
            pl.col("time").is_not_null() & pl.col("numeric_value").is_not_null()
        )
        .with_columns(pl.col("time").dt.truncate(scfg["agg"]).alias("time"))
        .sort(["subject_id", "code", "time"])
        .group_by(["subject_id", "code", "time"])
        .agg(
            [
                pl.col("numeric_value").min().alias("min"),
                pl.col("numeric_value").max().alias("max"),
                pl.col("prediction_time").last(),
                pl.col("boolean_value").last(),
            ]
        )
    )

    dynamic_numeric = (
        dynamic_numeric.unpivot(
            index=["subject_id", "code", "time", "prediction_time", "boolean_value"],
            on=["min", "max"],
            variable_name="stat",
            value_name="numeric_value",
        )
        .with_columns(
            pl.concat_str([pl.col("code"), pl.lit("_"), pl.col("stat")]).alias("code")
        )
        .drop("stat")
        .select(meds_core_columns)
    ).filter(pl.col("numeric_value").is_not_null())

    # dynamic_numeric = aggregate_events(dynamic_numeric, agg=scfg["agg"])

    dynamic_codes = events_base.filter(
        pl.col("time").is_not_null() & pl.col("numeric_value").is_null()
    )

    return static_codes, static_numeric, dynamic_numeric, dynamic_codes

In [ ]:
from pathlib import Path

from utils.preprocessing import generate_samples, filter_by_treshold, create_meds_cohort
import joblib
from meds2rdf import MedsRDFConverter, NTriplesSink, Config, MEDSSchema
from utils.preprocessing import PREFIX_MAP, process_codes

events = pl.scan_parquet("MEDS_cohort/data/**/*.parquet", low_memory=True).select(
    "subject_id", "time", "code", "numeric_value", "text_value"
)

for group_name, experiments in config["experiments"].items():
    print(f"\n=== Group: {group_name} ===")

    for scfg in experiments:
        print(f"Running task: {scfg['task']}")

        IHM = f"MEDS_cohort/labels/{scfg['task']}/**/*.parquet"

        outcomes = pl.scan_parquet(IHM).select(
            "subject_id", "prediction_time", "boolean_value"
        )

        samples = generate_samples(
            labels=outcomes,
            n_folds=scfg["num_of_samples"],
            size=scfg["sample_size"],
            seed=1234,
            low_true_values=scfg["fixed_true"],
        )

        TRESHOLD = scfg["sample_size"] / 5
        print("TRESHOLD: > ", TRESHOLD)

        for index, sample in enumerate(samples):
            export_dir, outcomes_dir, meds_cohort_dir = init_dirs(scfg["task"], index)

            events_base = build_events_base(events, sample, scfg)

            static_codes, static_numeric, dynamic_numeric, dynamic_codes = (
                split_event_types(events_base, scfg)
            )

            final_events = filter_by_treshold(
                pl.concat(
                    [
                        static_numeric,
                        static_codes,
                        dynamic_numeric,
                        dynamic_codes,
                    ],
                ),
                TRESHOLD,
            ).collect(engine="streaming")

            print(f"TOTAL EVENTS: {len(final_events)}")
            print(f"TOTAL CODES: {len(final_events.group_by('code').len())}")

            (_, _, split_l) = create_meds_cohort(
                final_events,
                orig_dir="MEDS_cohort",
                output_dir=meds_cohort_dir,
                columns=["subject_id", "code", "time", "numeric_value"],
            )

            joblib.dump(
                value=split_l.sort("subject_id")["boolean_value"].to_numpy(),
                filename=f"{outcomes_dir}/outcomes_meds_TS_{scfg['sample_size']}_{index}.joblib",
            )

            MedsRDFConverter(meds_cohort_dir).convert(
                sink=NTriplesSink(
                    Path(f"{export_dir}/meds_{scfg['sample_size']}_{index}"),
                    gzip_mode=False,
                ),
                cfg=Config(schemas={MEDSSchema.CODES}),
            )

            process_codes(
                input_parquet=f"{meds_cohort_dir}/metadata/codes.parquet",
                output_dir=f"{meds_cohort_dir}/mimic_external_codes",
                prefix_map=PREFIX_MAP,
            )

### e) Predict patient outcomes with tabular-based models

In [ ]:
from utils.tabular import run_tabulars_models
from collections import defaultdict
import yaml

CLASSES = ["Alive", "Dead"]

with open("experiments.yaml", "r") as f:
    config = yaml.safe_load(f)

feature_names_dict = defaultdict(list)
X_dict = defaultdict(list)

for group_name, experiments in config["experiments"].items():
    print(f"\n=== Group: {group_name} ===")

    for scfg in experiments:
        task = scfg["task"]
        print(f"Running task: {task}")

        ETL_LABELS = f"exports/{task}/labels"
        NUM_PATIENTS = scfg["sample_size"]

        for i in range(scfg["num_of_samples"]):
            EXPORT_DIR = f"exports/{task}/meds/{i}"
            X, y = run_tabulars_models(
                meds_root=f"{EXPORT_DIR}/MEDS_cohort",
                classes=CLASSES,
                outcomes_path=f"{ETL_LABELS}/outcomes_meds_TS_{NUM_PATIENTS}_{i}.joblib",
                result_dir=f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}",
                save_model=True
            )

            feature_names_dict[task].append(X.columns.to_list())
            X_dict[task].append(X)



In [ ]:
import joblib
joblib.dump(feature_names_dict, f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/feature_names_dict.joblib")
joblib.dump(X_dict, f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}/X_dict.joblib")


### Explain results through SHAP

In [ ]:
import shap
import joblib
from pathlib import Path 

model_name = "xgboost"
sample_idx = 0
task = "first_24_in_hospital_mortality"
NUM_PATIENTS = 500

feature_names_dict = joblib.load(f"exports/{task}/meds/{sample_idx}/metrics_{NUM_PATIENTS}/feature_names_dict.joblib")
X_dict = joblib.load(f"exports/{task}/meds/{sample_idx}/metrics_{NUM_PATIENTS}/X_dict.joblib")

model_dir = Path(
    f"exports/{task}/meds/{sample_idx}/metrics_{NUM_PATIENTS}/{model_name}/models"
)

best_model_path = str(
    max(
        model_dir.glob(f"{model_name}_best_fold*_auc_*.joblib"),
        key=lambda p: float(p.stem.split("_auc_")[-1]),
    )
)

explainer = shap.TreeExplainer(model=joblib.load(best_model_path))

In [ ]:
import shap
import matplotlib.pyplot as plt

X = X_dict[task][sample_idx]

shap_values = explainer.shap_values(X)

for class_idx, class_name in enumerate(CLASSES):
    shap.summary_plot(
        shap_values[:, :, class_idx],
        X,
        feature_names=feature_names_dict[task][sample_idx],
        show=False,
    )

    plt.savefig(
        f"exports/{task}/meds/{sample_idx}/metrics_{NUM_PATIENTS}/{model_name}/shap_class_{class_name}.png",  # type: ignore
        bbox_inches="tight",
        dpi=300,
    )
    plt.close()

In [ ]:
import numpy as np

X = X_dict[task][sample_idx]

shap_values = explainer.shap_values(X)

top_features_per_class = {}

for class_idx, class_name in enumerate(CLASSES):

    # SHAP values for this class
    class_shap = shap_values[:, :, class_idx]

    # Mean absolute importance per feature
    importance = np.abs(class_shap).mean(axis=0)

    # Sort descending
    sorted_idx = np.argsort(importance)[::-1]

    feature_names = np.array([
        f.replace("_count", "").replace("//", "_")
        for f in feature_names_dict[task][sample_idx]
    ])

    sorted_features = feature_names[sorted_idx]
    sorted_importance = importance[sorted_idx]

    # Store results
    top_features_per_class[class_name] = {
        "features": sorted_features,
        "importance": sorted_importance,
    }

    print(f"\nTop features for class {class_name}:")
    for f, v in zip(sorted_features[:30], sorted_importance[:30]):
        print(f"{f}: {v:.4f}")